# TP2 - Roubos de veiculos no RJ (2025 x 2024)

Notebook de conferencia: CSV, JSON/YAML, nota tecnica e tokens. Rode as celulas de cima para baixo, com o Jupyter aberto na pasta `codigo/`.

O que o notebook faz:

1. Le o CSV do ISP-RJ e monta o quadro por regiao e ano (gabarito do pandas, usado para conferir a IA).
2. Valida o JSON/YAML devolvido pelo chat (FASE 3) contra o gabarito, campo a campo, e lista cada divergencia (insumo para a tabela de correcoes).
3. Confere os numeros da nota (FASE 6) contra o CSV, o limite de 70 palavras do resumo e a ausencia de cifras da NT 14/2026.
4. Mede os tokens (nota modelo, CSV, prompts por fase e chat exportado) para o documento de previsao.

A nota tecnica (.docx) e o documento de previsao de tokens sao escritos no Word. O nome do autor entra so no Word, nao nos prompts.


In [1]:
# Configuracao: importa as bibliotecas e define ferramenta, modelo, data, caminhos, janela de contexto e funcoes auxiliares
import json
import random
import re
from pathlib import Path

import pandas as pd
import tiktoken
import yaml

# Ferramenta, modelo e data de uso do chat
ferramenta = "ChatGPT (OpenAI)"
modelo = "GPT-6 Astra"
data_uso = "09/11/2026"

caminho_csv = "../data/BaseDPEvolucaoMensalCisp.csv"
pasta_saidas = Path("../saidas")
pasta_entregaveis = Path("../entregaveis")
caminho_prompts = "../prompts/roteiro_prompts_crispdm.md"

# Janela de contexto do modelo, com a fonte oficial e a data da consulta
janela = 1050000
fonte_janela = "https://developers.openai.com/api/docs/models/gpt-6-astra"
data_consulta_janela = "09/11/2026"

# Palavras previstas para a nota (cabecalho, resumo, secoes 1 a 6, fonte), base da estimativa de tokens
palavras_previstas = 500

# Indicam se os prompts de contingencia foram usados no chat (so entao entram na previsao de tokens)
usou_correcao_fase3 = False   # FASE 3 (correcao)
usou_retorno_fase4 = False    # RETORNO A FASE 4

# Tokenizador: o200k_base (tiktoken). E uma aproximacao do tokenizador do modelo usado no chat.
enc = tiktoken.get_encoding("o200k_base")


# Formata um inteiro com ponto de milhar (30930 -> 30.930)
def formatar(n):
    return f"{int(n):,}".replace(",", ".")


# Variacao percentual de 'antes' para 'depois', com uma casa decimal
def variacao(antes, depois):
    return round((depois - antes) / antes * 100, 1)


# Formata um percentual no estilo da nota (-18.4 -> 18,4%)
def formatar_pct(x):
    return f"{abs(x):.1f}".replace(".", ",") + "%"


print(ferramenta, "|", modelo, "|", data_uso)
print("CSV existe?", Path(caminho_csv).exists())


ChatGPT (OpenAI) | GPT-6 Astra | 09/11/2026
CSV existe? True


## 1. Leitura do CSV

Separador ponto e virgula. Encoding latin-1 (nao UTF-8).


In [2]:
# Le o CSV do ISP-RJ, filtra 2024 e 2025 e confirma 12 meses por ano, todos na fase 3 (consolidada)
df = pd.read_csv(caminho_csv, sep=";", encoding="latin-1")
print("Linhas e colunas:", df.shape)

dados = df[df["ano"].isin([2024, 2025])].copy()
print("Linhas 2024 e 2025:", len(dados))

meses_2024 = sorted(dados[dados["ano"] == 2024]["mes"].unique().tolist())
meses_2025 = sorted(dados[dados["ano"] == 2025]["mes"].unique().tolist())
print("Meses 2024:", meses_2024)
print("Meses 2025:", meses_2025)
print(dados.groupby(["ano", "fase"]).size())

assert meses_2024 == list(range(1, 13))
assert meses_2025 == list(range(1, 13))
assert (dados["fase"] == 3).all()
print("Recorte ok: 12 meses em cada ano, todos na fase 3.")


Linhas e colunas: (38410, 65)
Linhas 2024 e 2025: 3288
Meses 2024: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Meses 2025: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
ano   fase
2024  3       1644
2025  3       1644
dtype: int64
Recorte ok: 12 meses em cada ano, todos na fase 3.


## 2. Grafia da regiao

Sem unificar, Grande Niterói vira duas regioes (uma com caracteres corrompidos) e o quadro passa de 8 para 9 linhas.


In [3]:
# Unifica as duas grafias de Grande Niterói (uma corrompida no arquivo) para o quadro ter 4 regioes
print("Regioes antes:")
print(dados["regiao"].value_counts())

dados.loc[dados["regiao"].str.startswith("Grande Niter"), "regiao"] = "Grande Niterói"

print("Regioes depois:")
print(dados["regiao"].value_counts())


Regioes antes:
regiao
Interior                         1608
Capital                           984
Baixada Fluminense                456
Grande Niterói                    150
Grande NiterÃÂÃÂÃÂÃÂ³i      90
Name: count, dtype: int64
Regioes depois:
regiao
Interior              1608
Capital                984
Baixada Fluminense     456
Grande Niterói         240
Name: count, dtype: int64


## 3. Quadro por regiao e ano (8 registros)

Totais da rubrica: roubos 30930 / 25235 e recuperados 19034 / 17406.


In [4]:
# Soma os quatro indicadores por regiao e ano (8 registros) e confere os totais do estado com os da rubrica
colunas = ["roubo_veiculo", "recuperacao_veiculos", "furto_veiculos", "roubo_carga"]
quadro = dados.groupby(["regiao", "ano"], as_index=False)[colunas].sum()
print(quadro.to_string(index=False))

assert len(quadro) == 8

# Totais do estado por ano (soma das quatro regioes)
totais = dados.groupby("ano")[colunas].sum()
print("Estado:")
print(totais)

assert int(totais.loc[2024, "roubo_veiculo"]) == 30930
assert int(totais.loc[2025, "roubo_veiculo"]) == 25235
assert int(totais.loc[2024, "recuperacao_veiculos"]) == 19034
assert int(totais.loc[2025, "recuperacao_veiculos"]) == 17406
print("Quadro conferido.")


            regiao  ano  roubo_veiculo  recuperacao_veiculos  furto_veiculos  roubo_carga
Baixada Fluminense 2024          10921                  6138            4451         1334
Baixada Fluminense 2025           8719                  5030            4504         1286
           Capital 2024          17302                  9960            8872         1769
           Capital 2025          13713                  9148            8680         1496
    Grande Niterói 2024           1939                  1264            1993          248
    Grande Niterói 2025           2058                  1367            2191          243
          Interior 2024            768                  1672            2021           86
          Interior 2025            745                  1861            2115           88
Estado:
      roubo_veiculo  recuperacao_veiculos  furto_veiculos  roubo_carga
ano                                                                   
2024          30930                 1903

## 4. JSON e YAML (gabarito do pandas)

O JSON e o YAML do ChatGPT sao colados na secao 6.


In [5]:
# Gera o gabarito do pandas em JSON e YAML (8 registros) e confirma que ambos sao validos e iguais ao quadro
pasta_saidas.mkdir(exist_ok=True)
registros = []
for _, linha in quadro.iterrows():
    registros.append({
        "regiao": str(linha["regiao"]),
        "ano": int(linha["ano"]),
        "roubo_veiculo": int(linha["roubo_veiculo"]),
        "recuperacao_veiculos": int(linha["recuperacao_veiculos"]),
        "furto_veiculos": int(linha["furto_veiculos"]),
        "roubo_carga": int(linha["roubo_carga"]),
    })

caminho_json = pasta_saidas / "quadro_regiao_ano.json"
caminho_yaml = pasta_saidas / "quadro_regiao_ano.yaml"
caminho_json.write_text(json.dumps(registros, ensure_ascii=False, indent=2), encoding="utf-8")
caminho_yaml.write_text(yaml.safe_dump(registros, allow_unicode=True, sort_keys=False), encoding="utf-8")

lido_json = json.loads(caminho_json.read_text(encoding="utf-8"))
lido_yaml = yaml.safe_load(caminho_yaml.read_text(encoding="utf-8"))
assert lido_json == registros
assert lido_yaml == registros
assert len(lido_json) == 8
print("JSON e YAML salvos com", len(lido_json), "registros.")


JSON e YAML salvos com 8 registros.


## 5. Variacoes, razao e Sao Goncalo


In [6]:
# Calcula as variacoes do estado, a razao recuperados/roubados, o roubo por regiao e Sao Goncalo,
# e monta a lista de numeros validos usada para conferir a nota
numeros_validos = []

# Variacao 2024 -> 2025 de cada indicador no estado
for col in colunas:
    v24 = int(totais.loc[2024, col])
    v25 = int(totais.loc[2025, col])
    dif = abs(v25 - v24)
    pct = variacao(v24, v25)
    print(col, formatar(v24), "->", formatar(v25), "| diferenca", formatar(dif), "|", pct, "%")
    numeros_validos += [formatar(v24), formatar(v25), formatar(dif), formatar_pct(pct)]

# Razao recuperados/roubados no estado (nao e taxa de recuperacao)
roubos_24 = int(totais.loc[2024, "roubo_veiculo"])
roubos_25 = int(totais.loc[2025, "roubo_veiculo"])
recup_24 = int(totais.loc[2024, "recuperacao_veiculos"])
recup_25 = int(totais.loc[2025, "recuperacao_veiculos"])
razao_2024 = round(recup_24 / roubos_24 * 100, 1)
razao_2025 = round(recup_25 / roubos_25 * 100, 1)
print("Razao recuperados/roubados:", razao_2024, "% /", razao_2025, "% (nao e taxa)")
numeros_validos += [formatar_pct(razao_2024), formatar_pct(razao_2025)]

# Variacao de cada indicador em cada regiao (imprime so o roubo de veiculos)
q = quadro.set_index(["regiao", "ano"])
regioes = ["Capital", "Baixada Fluminense", "Grande Niterói", "Interior"]
print("Roubo de veiculo por regiao:")
for regiao in regioes:
    for col in colunas:
        v24 = int(q.loc[(regiao, 2024), col])
        v25 = int(q.loc[(regiao, 2025), col])
        dif = abs(v25 - v24)
        pct = variacao(v24, v25)
        numeros_validos += [formatar(v24), formatar(v25), formatar(dif), formatar_pct(pct)]
        if col == "roubo_veiculo":
            print(regiao, formatar(v24), "->", formatar(v25), pct, "%")

# Roubos por municipio em Grande Niterói, para a participacao de Sao Goncalo na regiao
gn = dados[dados["regiao"] == "Grande Niterói"]
mun_gn = gn.groupby(["munic", "ano"])["roubo_veiculo"].sum()
print(mun_gn)
sg24 = int(mun_gn[("São Gonçalo", 2024)])
sg25 = int(mun_gn[("São Gonçalo", 2025)])
gn25 = int(q.loc[("Grande Niterói", 2025), "roubo_veiculo"])
print("São Gonçalo:", sg24, "em 2024 e", sg25, "em 2025, de", gn25, "casos da regiao em 2025")
numeros_validos += [formatar(sg24), formatar(sg25), formatar(abs(sg25 - sg24)), formatar_pct(variacao(sg24, sg25))]

print("Numeros validos para a nota:", len(numeros_validos))


roubo_veiculo 30.930 -> 25.235 | diferenca 5.695 | -18.4 %
recuperacao_veiculos 19.034 -> 17.406 | diferenca 1.628 | -8.6 %
furto_veiculos 17.337 -> 17.490 | diferenca 153 | 0.9 %
roubo_carga 3.437 -> 3.113 | diferenca 324 | -9.4 %
Razao recuperados/roubados: 61.5 % / 69.0 % (nao e taxa)
Roubo de veiculo por regiao:
Capital 17.302 -> 13.713 -20.7 %
Baixada Fluminense 10.921 -> 8.719 -20.2 %
Grande Niterói 1.939 -> 2.058 6.1 %
Interior 768 -> 745 -3.0 %
munic        ano 
Maricá       2024      43
             2025      63
Niterói      2024     338
             2025     279
São Gonçalo  2024    1558
             2025    1716
Name: roubo_veiculo, dtype: int64
São Gonçalo: 1558 em 2024 e 1716 em 2025, de 2058 casos da regiao em 2025
Numeros validos para a nota: 86


## 6. JSON e YAML do ChatGPT (FASE 3)

Cole os blocos devolvidos na FASE 3 (ou na FASE 3 de correcao). A celula valida o JSON e o YAML, compara todos os campos com o gabarito do CSV e lista cada divergencia, para registrar na tabela de correcoes.


In [7]:
# Valida o JSON/YAML devolvido pelo chat na FASE 3 contra o gabarito do CSV e lista cada divergencia
json_modelo = """
[
  {
    "regiao": "Capital",
    "ano": 2024,
    "roubo_veiculo": 17302,
    "recuperacao_veiculos": 9960,
    "furto_veiculos": 8872,
    "roubo_carga": 1769
  },
  {
    "regiao": "Capital",
    "ano": 2025,
    "roubo_veiculo": 13713,
    "recuperacao_veiculos": 9148,
    "furto_veiculos": 8680,
    "roubo_carga": 1496
  },
  {
    "regiao": "Baixada Fluminense",
    "ano": 2024,
    "roubo_veiculo": 10921,
    "recuperacao_veiculos": 6138,
    "furto_veiculos": 4451,
    "roubo_carga": 1334
  },
  {
    "regiao": "Baixada Fluminense",
    "ano": 2025,
    "roubo_veiculo": 8719,
    "recuperacao_veiculos": 5030,
    "furto_veiculos": 4504,
    "roubo_carga": 1286
  },
  {
    "regiao": "Grande Niterói",
    "ano": 2024,
    "roubo_veiculo": 1939,
    "recuperacao_veiculos": 1264,
    "furto_veiculos": 1993,
    "roubo_carga": 248
  },
  {
    "regiao": "Grande Niterói",
    "ano": 2025,
    "roubo_veiculo": 2058,
    "recuperacao_veiculos": 1367,
    "furto_veiculos": 2191,
    "roubo_carga": 243
  },
  {
    "regiao": "Interior",
    "ano": 2024,
    "roubo_veiculo": 768,
    "recuperacao_veiculos": 1672,
    "furto_veiculos": 2021,
    "roubo_carga": 86
  },
  {
    "regiao": "Interior",
    "ano": 2025,
    "roubo_veiculo": 745,
    "recuperacao_veiculos": 1861,
    "furto_veiculos": 2115,
    "roubo_carga": 88
  }
]
"""

yaml_modelo = """
- regiao: "Capital"
  ano: 2024
  roubo_veiculo: 17302
  recuperacao_veiculos: 9960
  furto_veiculos: 8872
  roubo_carga: 1769
- regiao: "Capital"
  ano: 2025
  roubo_veiculo: 13713
  recuperacao_veiculos: 9148
  furto_veiculos: 8680
  roubo_carga: 1496
- regiao: "Baixada Fluminense"
  ano: 2024
  roubo_veiculo: 10921
  recuperacao_veiculos: 6138
  furto_veiculos: 4451
  roubo_carga: 1334
- regiao: "Baixada Fluminense"
  ano: 2025
  roubo_veiculo: 8719
  recuperacao_veiculos: 5030
  furto_veiculos: 4504
  roubo_carga: 1286
- regiao: "Grande Niterói"
  ano: 2024
  roubo_veiculo: 1939
  recuperacao_veiculos: 1264
  furto_veiculos: 1993
  roubo_carga: 248
- regiao: "Grande Niterói"
  ano: 2025
  roubo_veiculo: 2058
  recuperacao_veiculos: 1367
  furto_veiculos: 2191
  roubo_carga: 243
- regiao: "Interior"
  ano: 2024
  roubo_veiculo: 768
  recuperacao_veiculos: 1672
  furto_veiculos: 2021
  roubo_carga: 86
- regiao: "Interior"
  ano: 2025
  roubo_veiculo: 745
  recuperacao_veiculos: 1861
  furto_veiculos: 2115
  roubo_carga: 88
"""

if not json_modelo.strip() or not yaml_modelo.strip():
    print("Cole o JSON e o YAML da FASE 3 para validar.")
else:
    # json.loads e yaml.safe_load geram erro se o texto nao for JSON/YAML valido
    jm = json.loads(json_modelo)
    ym = yaml.safe_load(yaml_modelo)
    print("Registros JSON:", len(jm), "| YAML:", len(ym))

    erros = []
    if len(jm) != 8:
        erros.append(f"JSON tem {len(jm)} registros (esperado: 8)")
    if len(ym) != 8:
        erros.append(f"YAML tem {len(ym)} registros (esperado: 8)")
    if jm != ym:
        erros.append("JSON e YAML nao sao iguais")

    # Compara cada registro do gabarito com o do modelo, campo a campo
    for nome, lista in [("JSON", jm), ("YAML", ym)]:
        for ref in registros:
            achado = None
            for item in lista:
                if item["regiao"] == ref["regiao"] and item["ano"] == ref["ano"]:
                    achado = item
            if achado is None:
                erros.append(f"{nome}: falta o registro {ref['regiao']} {ref['ano']}")
            else:
                for campo in ref:
                    if achado.get(campo) != ref[campo]:
                        erros.append(f"{nome}: {ref['regiao']} {ref['ano']} {campo} = {achado.get(campo)} (CSV: {ref[campo]})")

    # Totais do estado somados a partir do JSON do modelo
    tot_roubo_24 = 0
    tot_roubo_25 = 0
    tot_recup_24 = 0
    tot_recup_25 = 0
    for item in jm:
        if item["ano"] == 2024:
            tot_roubo_24 += item.get("roubo_veiculo", 0)
            tot_recup_24 += item.get("recuperacao_veiculos", 0)
        elif item["ano"] == 2025:
            tot_roubo_25 += item.get("roubo_veiculo", 0)
            tot_recup_25 += item.get("recuperacao_veiculos", 0)
    print("roubos 2024/2025:", tot_roubo_24, tot_roubo_25)
    print("recuperados 2024/2025:", tot_recup_24, tot_recup_25)
    if (tot_roubo_24, tot_roubo_25, tot_recup_24, tot_recup_25) != (30930, 25235, 19034, 17406):
        erros.append("os totais do estado nao conferem com 30930/25235 e 19034/17406")

    if len(erros) > 0:
        print("DIVERGENCIAS (registre cada uma na tabela de correcoes):")
        for erro in erros:
            print(" -", erro)
    else:
        print("JSON e YAML do modelo conferem com o CSV (8 registros, todos os campos e totais).")


Registros JSON: 8 | YAML: 8
roubos 2024/2025: 30930 25235
recuperados 2024/2025: 19034 17406
JSON e YAML do modelo conferem com o CSV (8 registros, todos os campos e totais).


## 7. Texto da nota (so para tokens)

Cole o texto final da FASE 6 em `texto_nota` (e o resumo em `texto_resumo`). A celula mede palavras e tokens, confere cada numero da nota contra o CSV, verifica se o resumo tem ate 70 palavras e so repete numeros do texto principal, e avisa se alguma cifra da NT 14/2026 apareceu. A nota em si e entregue em Word.


In [11]:
# Confere a nota final: mede palavras e tokens, checa os numeros contra o CSV, procura cifras da NT 14/2026
# e verifica o limite de 70 palavras do resumo executivo

# Texto final da nota (cabecalho, resumo, secoes 1 a 6 e fonte) e, separado, o resumo executivo, vindos da FASE 6
texto_nota = """
NOTA TÉCNICA INTERNA
Data: 9 de novembro de 2026
Autor(a): ________________________
Declaração de uso de IA: Texto redigido com apoio de inteligência artificial e conferido contra o CSV do ISP-RJ.

Assunto
roubos e recuperação de veículos no estado do Rio de Janeiro, 2025 em comparação com 2024

Resumo executivo
Em 2025, os roubos de veículos no estado diminuíram 18,4% em relação a 2024, enquanto as recuperações caíram 8,6%. Os furtos de veículos aumentaram 0,9%, e os roubos de carga diminuíram 9,4%. Grande Niterói foi a única região com aumento de roubos de veículos. A razão entre recuperados e roubados não corresponde a uma taxa de recuperação.

Contagem: 51 palavras.

1. Objetivo e método
Esta nota compara os registros de roubo de veículos, recuperação de veículos, furto de veículos e roubo de carga no estado do Rio de Janeiro em 2025 e 2024. São comparados os doze meses de cada ano, de janeiro a dezembro.

A fonte é a série mensal por área de delegacia do ISP-RJ. Os indicadores foram somados por região, com a grafia de Grande Niterói unificada. Os resultados estaduais são apresentados conforme o quadro conferido para esta análise.

2. Resultados no estado
Os roubos de veículos passaram de 30.930, em 2024, para 25.235, em 2025, uma redução de 5.695 casos (18,4%).

As recuperações de veículos passaram de 19.034 para 17.406, redução de 1.628 (8,6%). Os furtos de veículos passaram de 17.337 para 17.490, aumento de 153 (0,9%). Os roubos de carga passaram de 3.437 para 3.113, redução de 324 (9,4%).

3. Resultados por região
Na Capital, os roubos de veículos passaram de 17.302, em 2024, para 13.713, em 2025, uma redução de 3.589 casos (20,7%). Na Baixada Fluminense, passaram de 10.921 para 8.719, redução de 2.202 (20,2%).

Em Grande Niterói, os registros passaram de 1.939 para 2.058, aumento de 119 (6,1%). No Interior, passaram de 768 para 745, redução de 23 (3,0%). Grande Niterói foi a única região com aumento.

Em 2025, São Gonçalo registrou 1.716 dos 2.058 roubos de veículos de Grande Niterói, ou 83,4%.

4. Recuperação de veículos
A razão entre veículos recuperados e roubados no estado foi de 61,5% em 2024 e 69,0% em 2025. Essa razão não é uma taxa de recuperação: os veículos recuperados podem incluir veículos furtados e veículos de outro período ou região.

5. Limitações
Os dados representam ocorrências registradas e não necessariamente todos os crimes ocorridos. A base utilizada nesta análise não fornece dados de população ou de frota, portanto não permite calcular taxas por habitante ou por frota. A comparação descreve variações entre períodos, mas não permite atribuí-las a causas específicas.

6. Encaminhamentos
Antes da assinatura, conferir os valores e a redação final contra o CSV do ISP-RJ. Interpretar os resultados como comparação de registros entre períodos equivalentes, sem inferir causas ou tratar a razão entre recuperados e roubados como taxa de recuperação.

Fonte
INSTITUTO DE SEGURANÇA PÚBLICA (Rio de Janeiro). Estatísticas de segurança pública: série histórica mensal por área de delegacia desde 01/2003 (arquivo BaseDPEvolucaoMensalCisp.csv). Rio de Janeiro: ISP, [2026]. Disponível em: https://www.ispdados.rj.gov.br/estatistica.html. Acesso em: 21 out. 2026.

Lista de conferência dos inteiros
Estado — origem: linha ESTADO do quadro da fase 3.

Roubo de veículos: 2024 — 30.930; 2025 — 25.235.
Recuperação de veículos: 2024 — 19.034; 2025 — 17.406.
Furto de veículos: 2024 — 17.337; 2025 — 17.490.
Roubo de carga: 2024 — 3.437; 2025 — 3.113.
Diferenças entre 2025 e 2024: roubo de veículos — -5.695; recuperação — -1.628; furto — +153; roubo de carga — -324. Origem: cálculo a partir das linhas ESTADO do quadro da fase 3.
Roubo de veículos por região — origem: linhas regionais do quadro da fase 3.

Capital: 2024 — 17.302; 2025 — 13.713; diferença — -3.589.
Baixada Fluminense: 2024 — 10.921; 2025 — 8.719; diferença — -2.202.
Grande Niterói: 2024 — 1.939; 2025 — 2.058; diferença — +119.
Interior: 2024 — 768; 2025 — 745; diferença — -23.
São Gonçalo — origem: detalhe fornecido para a fase 4. Roubo de veículos em 2025 — 1.716. Grande Niterói, 2025 — 2.058, com origem na linha regional correspondente do quadro da fase 3.

Percentuais: foram calculados a partir dos inteiros indicados nesta lista; os valores não foram apresentados como dados de origem independentes.

Encontrado em:

Estatísticas de Segurança Pública - ISPdados Abertos
"""

texto_resumo = """
Em 2025, os roubos de veículos no estado diminuíram 18,4% em relação a 2024, enquanto as recuperações caíram 8,6%. Os furtos de veículos aumentaram 0,9%, e os roubos de carga diminuíram 9,4%. Grande Niterói foi a única região com aumento de roubos de veículos. A razão entre recuperados e roubados não corresponde a uma taxa de recuperação.
"""

# Cifras da NT 14/2026 (nota modelo), usadas para detectar copia delas na nota do TP2
numeros_nt14 = [
    "11.736", "13.430", "1.694", "8.658", "10.116", "1.574", "1.733", "8.417", "7.656",
    "6.614", "7.461", "4.027", "3.931", "333", "462", "762", "1.576", "1.396",
    "14,4%", "16,8%", "10,1%", "9,0%", "12,8%", "2,4%", "38,7%", "106,8%", "73,8%", "75,3%",
]


anos = ["2003", "2024", "2025", "2026"]


# Lista os numeros de um texto (inteiros e percentuais), ignorando anos, datas e numeros pequenos
def extrair_numeros(texto):
    numeros = []
    for palavra in texto.split():
        palavra = palavra.strip(".,;:()-")
        if palavra == "" or not palavra[0].isdigit():
            continue
        if "/" in palavra:
            continue
        if palavra.isdigit() and (len(palavra) < 3 or palavra in anos):
            continue
        numeros.append(palavra)
    return numeros


if not texto_nota.strip():
    pal_nota = None
    tok_nota = None
    print("Cole o texto da FASE 6 para medir palavras e tokens da nota e conferir os numeros.")
    print("A estimativa usa palavras_previstas =", palavras_previstas)
else:
    texto_nota = texto_nota.strip()
    pal_nota = len(texto_nota.split())
    tok_nota = len(enc.encode(texto_nota))
    print("Nota TP2 medida | palavras", pal_nota, "| tokens", tok_nota)

    numeros_nota = extrair_numeros(texto_nota)
    print("Numeros encontrados na nota:", len(numeros_nota))

    # Numeros da nota que nao estao entre os valores calculados do CSV
    fora = []
    for n in numeros_nota:
        if n not in numeros_validos and n not in fora:
            fora.append(n)
    print("Numeros que NAO estao no CSV:", fora if len(fora) > 0 else "nenhum")

    # Cifras da NT 14/2026 que apareceram na nota
    contaminados = []
    for n in numeros_nt14:
        if n in numeros_nota and n not in numeros_validos:
            contaminados.append(n)
    print("Cifras da NT 14/2026 na nota:", contaminados if len(contaminados) > 0 else "nenhuma")

    if texto_resumo.strip():
        resumo = texto_resumo.strip()
        pal_resumo = len(resumo.split())
        if pal_resumo <= 70:
            print("Resumo executivo:", pal_resumo, "palavras (limite 70) | OK")
        else:
            print("Resumo executivo:", pal_resumo, "palavras (limite 70) | ACIMA DO LIMITE")

        # Numeros do resumo que nao aparecem no texto principal (nota sem o resumo)
        texto_principal = texto_nota.replace(resumo, "")
        numeros_principal = extrair_numeros(texto_principal)
        fora_do_texto = []
        for n in extrair_numeros(resumo):
            if n not in numeros_principal:
                fora_do_texto.append(n)
        print("Numeros do resumo ausentes no texto principal:", fora_do_texto if len(fora_do_texto) > 0 else "nenhum")
    else:
        print("Cole o resumo executivo em texto_resumo para conferir o limite de 70 palavras.")


Nota TP2 medida | palavras 713 | tokens 1290
Numeros encontrados na nota: 63
Numeros que NAO estao no CSV: ['83,4%']
Cifras da NT 14/2026 na nota: nenhuma
Resumo executivo: 57 palavras (limite 70) | OK
Numeros do resumo ausentes no texto principal: nenhum


## 8. Previsao de tokens

Ferramenta de tokenizacao: `tiktoken`, tokenizador `o200k_base` (aproximacao do tokenizador do modelo do chat). A estimativa da nota usa `palavras_previstas`, definida no inicio. A saida prevista de cada fase vem de uma quantidade de palavras esperada vezes a razao tokens/palavra da nota modelo. Os prompts de contingencia (correcao da FASE 3 e retorno a FASE 4) so entram na previsao se as flags no inicio estiverem em `True`. Copie os numeros impressos para o Word de previsao de tokens.


In [9]:
# Mede palavras e tokens da nota modelo, estima os tokens da nota do TP2 e os tokens do CSV inteiro a partir de uma amostra

# Texto da NT 14/2026 exatamente como esta no enunciado (com acentos), para medir palavras e tokens
nota_modelo = """NOTA TÉCNICA INTERNA Nº 14/2026 (documento fictício, com números da base pública do ISP-RJ)
Assunto: roubos e recuperação de veículos no estado do Rio de Janeiro, 1º semestre de 2026.
1. Objetivo e método. Esta nota resume a evolução dos roubos de veículos, dos veículos recuperados, dos furtos de veículos e dos roubos de carga no estado do Rio de Janeiro, para apoiar a reunião de planejamento do segundo semestre. A comparação usa os meses de janeiro a junho de 2026 e os mesmos meses de 2025, de modo que os dois períodos tenham o mesmo tamanho e a mesma sazonalidade. Os dados vêm da série mensal por área de delegacia (CISP) do ISP-RJ. Os meses de julho a setembro de 2026 estão em outra fase de consolidação na base (fase 2) e ficaram fora da comparação.
2. Resultados no estado. Os roubos de veículos passaram de 11.736 para 13.430, um aumento de 1.694 casos (14,4%). Os veículos recuperados passaram de 8.658 para 10.116, um aumento de 16,8%. Os roubos de carga subiram de 1.574 para 1.733 (10,1%). Os furtos de veículos, ao contrário, caíram de 8.417 para 7.656 (queda de 9,0%).
3. Resultados por região. A Capital concentra o maior número de roubos de veículos: 6.614 em 2025 e 7.461 em 2026 (aumento de 12,8%). A Baixada Fluminense foi a única região com queda, de 4.027 para 3.931 (2,4%). O Interior passou de 333 para 462 (38,7%). A Grande Niterói passou de 762 para 1.576 (106,8%); o município de São Gonçalo responde por 1.396 dos 1.576 casos da região em 2026.
4. Recuperação de veículos. A razão entre veículos recuperados e veículos roubados no estado foi de 73,8% em 2025 e de 75,3% em 2026. Essa razão não é uma taxa de recuperação: os veículos recuperados incluem também veículos furtados e veículos roubados antes do período ou em outra região.
5. Limitações. A base traz registros de ocorrências, e não o total de crimes que de fato aconteceram. Não há informação sobre frota nem sobre população, de modo que não é possível calcular taxas por veículo ou por habitante. A base também não informa as causas das variações, e esta nota não as atribui a nenhuma política, operação ou mudança no registro das ocorrências.
6. Encaminhamentos. Recomenda-se: (a) verificar com a área responsável pelos registros a série de São Gonçalo antes de divulgar o resultado da Grande Niterói; (b) repetir a análise com os meses de julho a setembro quando a base for consolidada; (c) produzir um boletim em linguagem acessível ao público geral."""

# Palavras, tokens e razao tokens/palavra da nota modelo; estimativa de tokens da nota do TP2
pal_modelo = len(nota_modelo.split())
tok_modelo = len(enc.encode(nota_modelo))
razao_tp = round(tok_modelo / pal_modelo, 3)
est_nota = int(round(palavras_previstas * razao_tp))
print("Nota modelo | palavras", pal_modelo, "| tokens", tok_modelo, "| tokens/palavra", razao_tp)
print("Estimativa da nota TP2:", palavras_previstas, "x", razao_tp, "=", est_nota)
if pal_nota is None:
    print("Nota TP2 ainda nao colada na secao 7.")
else:
    print("Nota TP2 medida | palavras", pal_nota, "| tokens", tok_nota)

# Evidencia da tokenizacao: primeiros tokens da nota modelo
ids = enc.encode(nota_modelo)
print("Primeiros 12 tokens da nota modelo:", [enc.decode([i]) for i in ids[:12]])

# CSV: amostra aleatoria de 1000 linhas (semente fixa) para estimar caracteres por token
texto_csv = Path(caminho_csv).read_text(encoding="latin-1")
linhas_csv = texto_csv.splitlines()
cabecalho, corpo = linhas_csv[0], linhas_csv[1:]
random.seed(42)
amostra = "\n".join([cabecalho] + random.sample(corpo, 1000))
tok_amostra = len(enc.encode(amostra))
chars_amostra = len(amostra)
chars_por_token = round(chars_amostra / tok_amostra, 4)
est_csv = int(round(len(texto_csv) / chars_por_token))
tok_csv = len(enc.encode(texto_csv))
print("CSV inteiro | linhas", len(linhas_csv), "| chars", len(texto_csv))
print("CSV amostra 1000 linhas | chars", chars_amostra, "| tokens", tok_amostra, "| chars/token", chars_por_token)
print("CSV inteiro estimado", est_csv, "| medido", tok_csv, "| diferenca %", round((est_csv - tok_csv) / tok_csv * 100, 1))
print("CSV cabe na janela?", tok_csv < janela, "(", round(tok_csv / janela * 100, 1), "% da janela). O CSV NAO foi anexado: so cabecalho, amostra e quadro conferido entram nos prompts.")


Nota modelo | palavras 425 | tokens 668 | tokens/palavra 1.572
Estimativa da nota TP2: 500 x 1.572 = 786
Nota TP2 medida | palavras 713 | tokens 1290
Primeiros 12 tokens da nota modelo: ['NOT', 'A', ' T', 'É', 'CN', 'ICA', ' INTER', 'NA', ' Nº', ' ', '14', '/']
CSV inteiro | linhas 38411 | chars 7155488
CSV amostra 1000 linhas | chars 187754 | tokens 135001 | chars/token 1.3908
CSV inteiro estimado 5144872 | medido 5164659 | diferenca % -0.4
CSV cabe na janela? False ( 491.9 % da janela). O CSV NAO foi anexado: so cabecalho, amostra e quadro conferido entram nos prompts.


In [10]:
# Monta a previsao de tokens por fase do CRISP-DM, compara com a janela de contexto e,
# se o chat exportado existir, mede os tokens reais e a diferenca em relacao ao previsto

# Separa os prompts do roteiro (cada prompt esta dentro de um bloco de codigo)
roteiro = Path(caminho_prompts).read_text(encoding="utf-8")
blocos = [parte.strip() for i, parte in enumerate(roteiro.split("```")) if i % 2 == 1]
print("Prompts encontrados no roteiro:", len(blocos))

# Tokens do JSON + YAML do gabarito, usados como saida prevista da FASE 3
tok_json_yaml = len(enc.encode(caminho_json.read_text(encoding="utf-8")))
tok_json_yaml += len(enc.encode(caminho_yaml.read_text(encoding="utf-8")))

# Tabela por fase: tokens de entrada (prompt), saida prevista (palavras esperadas x tokens por palavra)
# e total acumulado no contexto
acum = 0
for bloco in blocos:
    titulo = bloco.splitlines()[0].strip()
    entrada = len(enc.encode(bloco))

    if titulo.startswith("FASE 1"):
        saida = int(450 * razao_tp)    # quatro blocos curtos, esqueleto em lista e vedacoes
    elif titulo.startswith("FASE 2"):
        saida = int(450 * razao_tp)    # colunas, riscos e recorte proposto
    elif titulo.startswith("FASE 3 (corre"):
        if not usou_correcao_fase3:
            print("(fora da previsao, contingencia nao usada):", titulo)
            continue
        saida = tok_json_yaml          # JSON + YAML corrigidos
    elif titulo.startswith("FASE 3"):
        saida = tok_json_yaml          # JSON + YAML
    elif titulo.startswith("FASE 4"):
        saida = int(500 * razao_tp)    # tres secoes em texto corrido e contas
    elif titulo.startswith("FASE 5"):
        saida = int(450 * razao_tp)    # tabela de conferencia e parecer
    elif titulo.startswith("RETORNO"):
        if not usou_retorno_fase4:
            print("(fora da previsao, contingencia nao usada):", titulo)
            continue
        saida = int(350 * razao_tp)    # secoes corrigidas e lista de mudancas
    else:
        saida = int((palavras_previstas + 120) * razao_tp)    # FASE 6: nota, resumo e lista de conferencia

    acum = acum + entrada + saida
    print(f"{titulo} | entrada {entrada} | saida prev. {saida} | acum. {acum}")

# Total previsto do chat e comparacao com a janela de contexto
prev_total = acum
print("Acumulado previsto:", prev_total, "de", janela, "|", round(prev_total / janela * 100, 2), "% da janela")
print("Chat inteiro cabe na janela?", prev_total < janela)
print("Fonte da janela:", fonte_janela, "| consulta:", data_consulta_janela)

# Procura o chat exportado em entregaveis/ e mede os tokens reais para comparar com o previsto
chat_path = None
for nome_arq in ["chat_exportado.md", "chat_exportado.txt", "chat_exportado.html"]:
    candidato = pasta_entregaveis / nome_arq
    if candidato.exists():
        chat_path = candidato
        break

if chat_path is None:
    print("Chat ainda nao exportado. Salve a exportacao em entregaveis/ como chat_exportado.md, .txt ou .html.")
    real_total = None
    pct_diff = None
else:
    chat_txt = chat_path.read_text(encoding="utf-8", errors="ignore")
    if chat_path.suffix == ".html":
        # remove as tags HTML para nao contar a marcacao como tokens do chat
        chat_txt = re.sub(r"<[^>]+>", " ", chat_txt)
    real_total = len(enc.encode(chat_txt))
    pct_diff = round((real_total - prev_total) / prev_total * 100, 1)
    print("Arquivo medido:", chat_path.name)
    print("Tokens reais:", real_total, "| previsto:", prev_total, "| diferenca %:", pct_diff)
    print("Explique a diferenca no documento (respostas mais longas ou curtas que o previsto, prompts de contingencia usados ou nao, texto extra da exportacao).")

print("Copie estes numeros para o documento de previsao de tokens (Word).")


Prompts encontrados no roteiro: 8
FASE 1: Entendimento do negócio | entrada 796 | saida prev. 707 | acum. 1503
FASE 2: Entendimento dos dados | entrada 983 | saida prev. 707 | acum. 3193
FASE 3: Preparação dos dados | entrada 959 | saida prev. 1010 | acum. 5162
(fora da previsao, contingencia nao usada): FASE 3 (correção): Preparação dos dados
FASE 4: Modelagem (análise) | entrada 923 | saida prev. 786 | acum. 6871
FASE 5: Avaliação | entrada 687 | saida prev. 707 | acum. 8265
(fora da previsao, contingencia nao usada): RETORNO À FASE 4: Modelagem (análise)
FASE 6: Implantação e comunicação | entrada 718 | saida prev. 974 | acum. 9957
Acumulado previsto: 9957 de 1050000 | 0.95 % da janela
Chat inteiro cabe na janela? True
Fonte da janela: https://developers.openai.com/api/docs/models/gpt-6-astra | consulta: 09/11/2026
Arquivo medido: chat_exportado.md
Tokens reais: 11363 | previsto: 9957 | diferenca %: 14.1
Explique a diferenca no documento (respostas mais longas ou curtas que o previs